In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pint
import pint_xarray
import xarray as xr

## Loading


In [ ]:
seapopym_v1 = xr.open_zarr(
    "/Users/adm-lehodey/Documents/Workspace/Projects/phd_optimization/notebooks/Article_1/data/2_global_simulation/biomass_global.zarr"
)
seapopym_v1 = (
    seapopym_v1["biomass"]
    .squeeze()
    .rename({"T": "time", "X": "x", "Y": "y"})
    .sel(time=slice("2000", "2019"))
    .load()
)
seapopym_v1.x.attrs = {}
seapopym_v1.y.attrs = {}
seapopym_v1 = seapopym_v1.pint.quantify().pint.to("g/m^2").pint.dequantify()

In [ ]:
seapopym_v2 = xr.open_zarr("./SeapoPym_Global_No_transport_v2.zarr")
seapopym_v2 = seapopym_v2.sel(time=slice("2000", "2019")).load()["biomass"]

In [ ]:
seapodym = xr.open_zarr(
    "/Users/adm-lehodey/Documents/Workspace/Projects/phd_optimization/notebooks/Article_1/data/1_global/post_processed_light_global_multiyear_bgc_001_033.zarr/"
)
seapodym = (
    seapodym["zooplankton"]
    .squeeze()
    .rename({"T": "time", "X": "x", "Y": "y"})
    .sel(time=slice("2000", "2019"))
    .load()
)
seapodym.x.attrs = {}
seapodym.y.attrs = {}
seapodym.attrs["units"] = "g/m^2"
seapodym = seapodym.assign_coords(time=seapodym.time.dt.floor("D"))

## Comparison between SeapoPym V1 and V2


In [ ]:
# Normaliser les timestamps pour l'alignement
# Option 1: Arrondir les deux à la date (floor à minuit)
seapopym_v1_normalized = seapopym_v1.assign_coords(time=seapopym_v1.time.dt.floor("D"))
seapopym_v2_normalized = seapopym_v2.assign_coords(time=seapopym_v2.time.dt.floor("D"))

# Vérifier l'alignement
print(
    f"v1 time range: {seapopym_v1_normalized.time.values[0]} to {seapopym_v1_normalized.time.values[-1]}"
)
print(
    f"v2 time range: {seapopym_v2_normalized.time.values[0]} to {seapopym_v2_normalized.time.values[-1]}"
)
print(f"v1 shape: {seapopym_v1_normalized.shape}")
print(f"v2 shape: {seapopym_v2_normalized.shape}")

# Calculer la différence
mape = np.abs((seapopym_v1_normalized - seapopym_v2_normalized) / seapopym_v1_normalized) * 100
mape.attrs["units"] = "% of v1"

In [ ]:
mape.mean("time").plot(vmin=0, vmax=10, cmap="viridis_r")

## Comparison between Seapopym and Seapodym-LMTL


In [ ]:
# Calculer la différence
mape_v1_original = np.abs((seapodym - seapopym_v1_normalized) / seapodym) * 100
mape_v1_original.attrs["units"] = "%"
mape_v2_original = np.abs((seapodym - seapopym_v2_normalized) / seapodym) * 100
mape_v2_original.attrs["units"] = "%"

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
mape_v1_original.mean("time").plot(ax=axes[0, 0], vmax=25)
axes[0, 0].set_title("v1")
mape_v2_original.mean("time").plot(ax=axes[0, 1], vmax=25)
axes[0, 1].set_title("v2")
np.abs(mape_v1_original - mape_v2_original).mean("time").plot(ax=axes[1, 0], vmax=10)
axes[1, 0].set_title("v1 - v2")
axes[1, 1].set_visible(False)